In [29]:
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
import torchvision.transforms as transforms
from torch.utils import data
import matplotlib.pyplot as plt

In [30]:
class Accumulator:
    def __init__(self, n):
        self.data = [0.0] * n
    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]
    def reset(self):
        self.data = [0.0] * len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

In [31]:
class Residual(nn.Module):
    def __init__(self, input_channels, num_channels, use_1x1conv = False, strides = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels, kernel_size = 3, padding = 1, stride = strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size = 3, padding = 1)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels, kernel_size = 1, stride = strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

In [32]:
b1 = nn.Sequential(nn.Conv2d(3, 64, kernel_size = 7, stride = 2, padding = 3),
                 nn.BatchNorm2d(64), nn.ReLU(),
                 nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1))

In [33]:
def resnet_block(input_channels, num_channels, num_residuals, 
                 first_block = False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(input_channels, num_channels, 
                                use_1x1conv = True, strides = 2))
        else:
            blk.append(Residual(num_channels, num_channels))
    return blk
        

In [34]:
b2 = nn.Sequential(*resnet_block(64, 64, 2, first_block = True))
b3 = nn.Sequential(*resnet_block(64, 128, 2))
b4 = nn.Sequential(*resnet_block(128, 256, 2))
b5 = nn.Sequential(*resnet_block(256, 512, 2))

In [35]:
net = nn.Sequential(b1, b2, b3, b4, b5,
                   nn.AdaptiveAvgPool2d((1, 1)),
                   nn.Flatten(), nn.Linear(512, 10))

In [44]:
def load_data(batch_size, resize = None):
    trans = []
    if resize:
        trans.append(transforms.Resize((resize, resize)))
    trans.append(transforms.ToTensor())
    trans.append(transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))
    trans = transforms.Compose(trans)
    
    train_dir = '/kaggle/input/datasets/tongpython/cat-and-dog/training_set/training_set'
    test_dir = '/kaggle/input/datasets/tongpython/cat-and-dog/test_set/test_set'
    full_dataset = torchvision.datasets.ImageFolder(root = train_dir, transform = trans)
    #划分训练集和验证集
    val_size = int(len(full_dataset) * 0.2)
    train_size = len(full_dataset) - val_size
    train_dataset, val_dataset = data.random_split(full_dataset, [train_size, val_size])
    #返回
    train_loader = data.DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
    val_loader = data.DataLoader(val_dataset, batch_size = batch_size, shuffle = False)
    return train_loader, val_loader


# 训练函数

In [45]:
#返回正确数
def accuracy(y_hat, y):
    if y_hat.ndim > 1:
        y_hat = y_hat.argmax(dim=1)
    return (y_hat == y).sum().item()

def evaluate_accuracy_gpu(net, data_iter, device = None):
    if isinstance(net, nn.Module):
        net.eval()
        if not device:
            device = next(iter(net.parameters())).device
    metric = Accumulator(2)
    with torch.no_grad():
        for X, y in data_iter:
            if isinstance(X, list):
                X =[x.to(device) for x in X]
            else:
                X = X.to(device)

            y = y.to(device)
            metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

In [46]:
def train(net, train_iter, test_iter, num_epochs, lr, device):
    #使用xavier权重初始化
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)
    print('training on', device)
    net.to(device)
    optimizer = torch.optim.SGD(net.parameters(), lr = lr)
    loss = nn.CrossEntropyLoss()
    

    for epoch in range(num_epochs):
        metric= Accumulator(3)
        for i,(x, y) in enumerate(train_iter):
            optimizer.zero_grad()
            x, y = x.to(device), y.to(device)
            y_hat = net(x)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()
            with torch.no_grad():
                metric.add(l * x.shape[0], accuracy(y_hat, y), x.shape[0])
            train_l = metric[0] / metric[2]
            train_acc = metric[1] / metric[2]
            
        test_acc = evaluate_accuracy_gpu(net, test_iter)

        print(f'Epoch {epoch+1}: loss={train_l:.4f}, train_acc={train_acc:.4f}, test_acc={test_acc:.4f}')
        

In [47]:
lr, num_epochs, batch_size = 0.0005, 10, 256
train_iter, test_iter = load_data(batch_size, resize = 96)
train(net, train_iter, test_iter, num_epochs, lr, torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

training on cuda
Epoch 1: loss=2.0668, train_acc=0.3840, test_acc=0.4997
Epoch 2: loss=0.8683, train_acc=0.5395, test_acc=0.5515
Epoch 3: loss=0.7116, train_acc=0.5676, test_acc=0.5053
Epoch 4: loss=0.6975, train_acc=0.5896, test_acc=0.5303
Epoch 5: loss=0.6677, train_acc=0.6163, test_acc=0.5134
Epoch 6: loss=0.6597, train_acc=0.6235, test_acc=0.4947
Epoch 7: loss=0.6559, train_acc=0.6402, test_acc=0.5703
Epoch 8: loss=0.6344, train_acc=0.6568, test_acc=0.6040
Epoch 9: loss=0.6252, train_acc=0.6705, test_acc=0.4978
Epoch 10: loss=0.6275, train_acc=0.6676, test_acc=0.5141
